# Máquina de Soporte Vectorial para el dataset *digits* de Scikit-learn
### Inteligencia Artificial ELP 8012 - Universidad del Norte - Profesor: Eduardo Zurek, Ph.D.
### **Grupo:** Natalia Carpintero, Paula Núñez e Isabella Arrieta.

Se implementa una Máquina de Soporte Vectorial (SVM) que clasifica los dígitos escritos a mano del dataset `digits`.

Pasos de la solución:
1. Cargar los datos.
2. Dividirlos en entrenamiento (80%) y prueba (20%).
3. Definir la SVM y los kernels a comparar.
4. Encontrar el mejor kernel con K-fold.
5. Evaluar el kernel elegido con los datos de prueba.
6. Comparar con los resultados de los talleres 9 y 10.

## 1. Carga de los datos

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data
y = digits.target
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("Clases:", np.unique(y))
print("Rango de los píxeles:", X.min(), "a", X.max())
print("Imágenes por clase:", np.bincount(y))

Forma de X: (1797, 64)
Forma de y: (1797,)
Clases: [0 1 2 3 4 5 6 7 8 9]
Rango de los píxeles: 0.0 a 16.0
Imágenes por clase: [178 182 177 183 181 182 181 179 174 180]


El dataset tiene 1797 imágenes de 8x8 píxeles. Cada píxel es una característica (64 en total) con valor entero entre 0 y 16, y la clase es el dígito (0 a 9). Las 10 clases están casi balanceadas.

## 2. División en entrenamiento y prueba

Se usa la misma división del taller 9 (80/20, `stratify=y`, `random_state=4`), para poder comparar los resultados. Los datos de prueba se reservan para la evaluación final.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(X, y, test_size=0.20, random_state=4, stratify=y)
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (1437, 64)
Prueba: (360, 64)


## 3. La SVM y los kernels

Una SVM busca el hiperplano que separa las clases con el mayor margen posible. El parámetro $C$ controla cuánto se penalizan los puntos que violan ese margen: con $C$ grande se toleran menos violaciones.

Cuando las clases no se pueden separar con un hiperplano en el espacio original, un **kernel** calcula el producto escalar en un espacio de mayor dimensión, donde pueden llegar a separarse. Se comparan los cuatro kernels vistos en clase:

| Kernel | Fórmula | Hiperparámetros propios |
|---|---|---|
| `linear` | $K(x_i,x_j)=x_i^T x_j$ | ninguno |
| `poly` | $K(x_i,x_j)=(\gamma x_i^T x_j+r)^d$ | $\gamma$, $r$ (`coef0`), $d$ (`degree`) |
| `rbf` | $K(x_i,x_j)=\exp(-\gamma\lVert x_i-x_j\rVert^2)$ | $\gamma$ |
| `sigmoid` | $K(x_i,x_j)=\tanh(\gamma x_i^T x_j+r)$ | $\gamma$, $r$ (`coef0`) |

Se usa `SVC` de Scikit-learn, igual que en el cuaderno del curso `05_SVM_SKLearn_Cuatro_Categorias_2_dimensiones.ipynb`. Una SVM separa solo dos clases, así que para los 10 dígitos `SVC` entrena una SVM por cada par de dígitos (45 en total) y predice el dígito que recibe más votos.

## 4. Búsqueda del mejor kernel con K-fold

Un kernel puede quedar mal solo porque sus hiperparámetros no están ajustados. Por eso se prueban varias combinaciones para cada kernel con `GridSearchCV` y se compara la mejor de cada uno.

* **K-fold:** k = 4 sobre los datos de entrenamiento, con las mismas particiones del taller 9. Se entrena con 3 partes, se valida con la restante y se promedia.
* **Indicadores:** accuracy, precision, recall y F1, con promedio `macro` entre las 10 clases. El mejor de cada kernel se elige por accuracy promedio.
* **$C$:** de 0.001 a 1000, en todos los kernels.
* **$\gamma$:** `'scale'` y de $10^{-5}$ a $10^{-2}$, en `rbf` y `sigmoid`. `'scale'` equivale a $\frac{1}{64\cdot Var(X)}$. En `poly` se deja `'scale'` y se varían $d$ (2 a 5) y $r$ (0 y 1), para no agrandar la búsqueda.
* **$r$ en `sigmoid`:** -1 y 0.
* **Estandarización:** se prueba con y sin `StandardScaler`, porque los píxeles ya están en la misma escala (0 a 16) y no es evidente que ayude. Va dentro del `Pipeline`, así el escalador se ajusta solo con las particiones de entrenamiento.

Los valores por defecto de `SVC` ($C=1$, `gamma='scale'`, $d=3$, $r=0$) están incluidos en la búsqueda.

In [3]:
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score

kfold = KFold(n_splits=4, shuffle=True, random_state=4)

# zero_division=0: con valores extremos de C o gamma, un modelo puede no predecir algún dígito
# y su precision queda indefinida; en ese caso se toma como 0.
indicadores = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall': make_scorer(recall_score, average='macro', zero_division=0),
    'f1': make_scorer(f1_score, average='macro', zero_division=0),
}

valores_C = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
valores_gamma = ['scale', 1e-5, 1e-4, 1e-3, 1e-2]
escalados = ['passthrough', StandardScaler()]  # 'passthrough' = sin estandarizar

parametros = {
    'linear':  {'escalador': escalados, 'svm__C': valores_C},
    'poly':    {'escalador': escalados, 'svm__C': valores_C,
                'svm__degree': [2, 3, 4, 5], 'svm__coef0': [0, 1]},
    'rbf':     {'escalador': escalados, 'svm__C': valores_C, 'svm__gamma': valores_gamma},
    'sigmoid': {'escalador': escalados, 'svm__C': valores_C, 'svm__gamma': valores_gamma,
                'svm__coef0': [-1, 0]},
}

busquedas = {}
filas = []
for kernel, rejilla in parametros.items():
    modelo = Pipeline([('escalador', 'passthrough'), ('svm', SVC(kernel=kernel))])
    busqueda = GridSearchCV(modelo, rejilla, cv=kfold, scoring=indicadores,
                            refit='accuracy', n_jobs=-1)
    busqueda.fit(X_train, y_train)
    busquedas[kernel] = busqueda

    i = busqueda.best_index_
    r = busqueda.cv_results_
    mejores = busqueda.best_params_
    filas.append({
        'Kernel': kernel,
        'accuracy': r['mean_test_accuracy'][i],
        'desv. accuracy': r['std_test_accuracy'][i],
        'precision': r['mean_test_precision'][i],
        'recall': r['mean_test_recall'][i],
        'F1': r['mean_test_f1'][i],
        'Estandarizado': mejores['escalador'] != 'passthrough',
        'Hiperparámetros': {k.replace('svm__', ''): v for k, v in mejores.items() if k != 'escalador'},
    })

pd.set_option('display.max_colwidth', None)
resultados = pd.DataFrame(filas).set_index('Kernel').sort_values('accuracy', ascending=False)
resultados.round(4)

,accuracy,desv. accuracy,precision,recall,F1,Estandarizado,Hiperparámetros
Kernel,,,,,,,
rbf,0.9882,0.0063,0.9888,0.9885,0.9885,False,"{'C': 10, 'gamma': 'scale'}"
poly,0.9861,0.0081,0.9866,0.9866,0.9864,True,"{'C': 1, 'coef0': 1, 'degree': 5}"
sigmoid,0.9854,0.0128,0.9860,0.9863,0.9858,False,"{'C': 100, 'coef0': -1, 'gamma': 0.0001}"
linear,0.9798,0.0087,0.9804,0.9806,0.9801,False,{'C': 0.01}


### Elección del kernel

El mejor kernel es **RBF** ($C=10$, `gamma='scale'`, sin estandarizar), con el mayor valor en los cuatro indicadores (accuracy 0.9882, precision 0.9888, recall 0.9885, F1 0.9885) y la menor desviación entre particiones (0.0063).

* **`poly`** (0.9861) y **`sigmoid`** (0.9854) quedan muy cerca: 3 y 4 aciertos menos que RBF de 1437 imágenes. Esa diferencia es menor que la desviación entre particiones, así que RBF es el mejor en esta evaluación, pero con una ventaja pequeña.
* **`linear`** (0.9798) es el más bajo, con 12 aciertos menos que RBF. Los dígitos son casi separables linealmente, y los kernels no lineales mejoran un poco el resultado.
* Estandarizar solo fue mejor en `poly`. En los otros tres kernels fue mejor dejar los píxeles en su escala original.

## 5. Evaluación del kernel elegido con los datos de prueba

`GridSearchCV` reentrena la mejor combinación con todos los datos de entrenamiento. Solo se evalúa el kernel elegido, de modo que los datos de prueba no influyen en ninguna decisión.

In [4]:
from sklearn.metrics import accuracy_score, confusion_matrix

mejor_kernel = resultados.index[0]
mejor_svm = busquedas[mejor_kernel].best_estimator_  # ya reentrenado con todo X_train

y_esperado = y_test
y_predicho = mejor_svm.predict(X_test)

print("Kernel elegido:", mejor_kernel)
print("accuracy :", round(accuracy_score(y_esperado, y_predicho), 4))
print("precision:", round(precision_score(y_esperado, y_predicho, average='macro'), 4))
print("recall   :", round(recall_score(y_esperado, y_predicho, average='macro'), 4))
print("F1       :", round(f1_score(y_esperado, y_predicho, average='macro'), 4))
print("Errores  :", (y_esperado != y_predicho).sum(), "de", len(y_test))

digitos = range(10)
pd.DataFrame(confusion_matrix(y_esperado, y_predicho),
             index=[f'real {d}' for d in digitos],
             columns=[f'pred {d}' for d in digitos])

Kernel elegido: rbf
accuracy : 0.9917
precision: 0.9921
recall   : 0.9916
F1       : 0.9916
Errores  : 3 de 360


,pred 0,pred 1,pred 2,pred 3,pred 4,pred 5,pred 6,pred 7,pred 8,pred 9
real 0,36,0,0,0,0,0,0,0,0,0
real 1,0,37,0,0,0,0,0,0,0,0
real 2,0,0,35,0,0,0,0,0,0,0
real 3,0,0,1,36,0,0,0,0,0,0
real 4,0,0,0,0,36,0,0,0,0,0
real 5,0,0,0,0,0,36,0,0,0,0
real 6,0,0,0,0,0,0,36,0,0,0
real 7,0,0,0,0,0,0,0,36,0,0
real 8,0,2,0,0,0,0,0,0,33,0
real 9,0,0,0,0,0,0,0,0,0,36


Los resultados en prueba (accuracy 0.9917, precision 0.9921, recall 0.9916, F1 0.9916) son similares a los de K-fold, por lo que el modelo generaliza bien.

En la matriz de confusión solo hay 3 errores en 360 imágenes: dos 8 clasificados como 1 y un 3 clasificado como 2.

## 6. Comparación con los talleres 9 y 10

### 6.1 Resultados tal como se obtuvieron en cada taller

* **Taller 9:** clasificador Bayesiano Categórico con $\alpha=1$, K-fold = 4 y la misma división 80/20 de este cuaderno.
* **Taller 10:** red neuronal MLP con activación `logistic`. No usó K-fold, sino una división 60/20/20 con `random_state=42`, por lo que entrenó con menos imágenes (1077) y su conjunto de prueba es otro.

In [5]:
# Valores copiados de las salidas guardadas en los cuadernos de cada taller
reportados = pd.DataFrame([
    {'Modelo': 'Taller 9: Bayes Categórico', 'Evaluación interna': 'K-fold = 4',
     'accuracy interna': 0.9116, 'accuracy prueba': 0.9167,
     'precision prueba': 0.9246, 'recall prueba': 0.9165},
    {'Modelo': 'Taller 10: MLP logistic', 'Evaluación interna': 'Validación simple',
     'accuracy interna': 0.9750, 'accuracy prueba': 0.9611,
     'precision prueba': 0.9614, 'recall prueba': 0.9607},
    {'Modelo': f'SVM kernel {mejor_kernel}', 'Evaluación interna': 'K-fold = 4',
     'accuracy interna': resultados.loc[mejor_kernel, 'accuracy'],
     'accuracy prueba': accuracy_score(y_esperado, y_predicho),
     'precision prueba': precision_score(y_esperado, y_predicho, average='macro'),
     'recall prueba': recall_score(y_esperado, y_predicho, average='macro')},
]).set_index('Modelo')
reportados.round(4)

,Evaluación interna,accuracy interna,accuracy prueba,precision prueba,recall prueba
Modelo,,,,,
Taller 9: Bayes Categórico,K-fold = 4,0.9116,0.9167,0.9246,0.9165
Taller 10: MLP logistic,Validación simple,0.9750,0.9611,0.9614,0.9607
SVM kernel rbf,K-fold = 4,0.9882,0.9917,0.9921,0.9916


### 6.2 Comparación con las mismas particiones

Como el taller 10 no usó K-fold y sus datos de entrenamiento y prueba son diferentes, la tabla anterior no compara a los modelos en igualdad de condiciones. Por eso se reentrena el modelo ganador de cada taller, con su misma configuración, usando las mismas 4 particiones y los mismos datos de prueba de la SVM.

In [6]:
from sklearn.model_selection import cross_validate
from sklearn.naive_bayes import CategoricalNB
from sklearn.neural_network import MLPClassifier

modelos = {
    'Taller 9: Bayes Categórico': CategoricalNB(alpha=1.0, min_categories=17),
    'Taller 10: MLP logistic': Pipeline([
        ('escalador', StandardScaler()),
        ('mlp', MLPClassifier(hidden_layer_sizes=(64,), activation='logistic', solver='lbfgs',
                              alpha=1e-4, tol=1e-6, max_iter=2000, max_fun=50000,
                              random_state=42))]),
    f'SVM kernel {mejor_kernel}': mejor_svm,
}

comparacion = []
errores = {}
for nombre, modelo in modelos.items():
    cv = cross_validate(modelo, X_train, y_train, cv=kfold, scoring=indicadores)
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    comparacion.append({
        'Modelo': nombre,
        'accuracy (K-fold)': cv['test_accuracy'].mean(),
        'desv. (K-fold)': cv['test_accuracy'].std(),
        'precision (K-fold)': cv['test_precision'].mean(),
        'recall (K-fold)': cv['test_recall'].mean(),
        'F1 (K-fold)': cv['test_f1'].mean(),
        'accuracy (prueba)': accuracy_score(y_test, y_pred),
        'precision (prueba)': precision_score(y_test, y_pred, average='macro'),
        'recall (prueba)': recall_score(y_test, y_pred, average='macro'),
        'F1 (prueba)': f1_score(y_test, y_pred, average='macro'),
    })
    errores[nombre] = int((y_test != y_pred).sum())

print("Errores en prueba (de 360):", errores)
comparacion = pd.DataFrame(comparacion).set_index('Modelo')
comparacion.round(4).T

Errores en prueba (de 360): {'Taller 9: Bayes Categórico': 30, 'Taller 10: MLP logistic': 10, 'SVM kernel rbf': 3}


Modelo,Taller 9: Bayes Categórico,Taller 10: MLP logistic,SVM kernel rbf
accuracy (K-fold),0.9116,0.9687,0.9882
desv. (K-fold),0.0216,0.0091,0.0063
precision (K-fold),0.9146,0.9690,0.9888
recall (K-fold),0.9164,0.9693,0.9885
F1 (K-fold),0.9124,0.9685,0.9885
accuracy (prueba),0.9167,0.9722,0.9917
precision (prueba),0.9246,0.9732,0.9921
recall (prueba),0.9165,0.9719,0.9916
F1 (prueba),0.9170,0.9715,0.9916


El Bayesiano reproduce exactamente los valores del taller 9 (0.9116 en K-fold y 0.9167 en prueba), porque la división es la misma. La MLP pasa de 0.9611 a 0.9722 en prueba al entrenar con 1437 imágenes en lugar de 1077 y evaluarse con otro conjunto de prueba. Por eso era necesario usar las mismas particiones para comparar.

| Modelo | Accuracy (K-fold) | Accuracy (prueba) | Errores en prueba |
|---|---|---|---|
| **SVM RBF** | **0.9882** | **0.9917** | **3 de 360** |
| MLP `logistic` (taller 10) | 0.9687 | 0.9722 | 10 de 360 |
| Bayes Categórico (taller 9) | 0.9116 | 0.9167 | 30 de 360 |

* La **SVM RBF** obtiene los mejores valores en todos los indicadores, tanto en K-fold como en prueba.
* Frente al **Bayesiano**, la diferencia es grande (27 errores menos en prueba). El Bayesiano ingenuo supone que los píxeles son independientes dada la clase, mientras que la SVM y la MLP sí pueden aprovechar las relaciones entre píxeles.
* Frente a la **MLP**, la diferencia es menor (7 errores menos en prueba y 1.95 puntos en K-fold), pero supera la desviación entre particiones de ambos modelos (0.0063 y 0.0091).

**Precaución:** los hiperparámetros de la SVM se eligieron con las mismas particiones de K-fold, lo que hace su promedio un poco optimista, y la MLP no tuvo una búsqueda equivalente. El resultado en prueba, que no se usó para ninguna decisión, también favorece a la SVM, pero parte de la ventaja frente a la MLP puede deberse a ese ajuste.

## 7. Conclusión

El kernel con los mejores indicadores para clasificar el dataset `digits` es **RBF** ($C=10$, `gamma='scale'`, sin estandarizar), con accuracy promedio de 0.9882 en K-fold (k = 4) y 0.9917 en prueba. Con las mismas particiones, supera a la MLP del taller 10 (0.9722 en prueba) y al clasificador Bayesiano del taller 9 (0.9167 en prueba).

## Referencias

- Material del curso: [`_009_SVM_maquinas_de_soporte_vectorial`](https://github.com/eduardozurek/ia/tree/main/_009_SVM_maquinas_de_soporte_vectorial).
- [Taller 9: Clasificadores Bayesianos](https://github.com/Inteligencia-Artificial-202630/Taller9-ClasificadoresBayesianos).
- [Taller 10: Redes Neuronales Artificiales](https://github.com/Inteligencia-Artificial-202630/Taller10-Redes-Neuronales-Artificiales).
- [`SVC`](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html) y [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) en Scikit-learn.